# 02 — Data Cleaning Analysis

This notebook applies cleaning steps to the raw CICIDS2017 dataset:
- Drop exact-duplicate rows
- Handle missing (NaN) values
- Remove infinite values
- Standardize label strings
- Produce a cleaned dataset ready for feature engineering

In [ ]:
import pandas as pd
import numpy as np
import os
import sys

sys.path.insert(0, os.path.join(os.path.dirname("__file__" if "__file__" in dir() else ""), ".."))

In [ ]:
RAW_PATH = os.path.join("..", "data", "raw", "CICIDS2017", "Wednesday-workingHours.pcap_ISCX.csv")
df = pd.read_csv(RAW_PATH)
print(f"Raw shape: {df.shape}")
df.head(3)

## 1. Drop Duplicates

In [ ]:
before = len(df)
df_clean = df.drop_duplicates().reset_index(drop=True)
removed = before - len(df_clean)
print(f"Rows before:  {before:,}")
print(f"Rows after:   {len(df_clean):,}")
print(f"Duplicates removed: {removed:,} ({removed / before * 100:.2f}%)")

## 2. Handle Missing Values

In [ ]:
null_before = df_clean.isnull().sum().sum()
print(f"Total NaN before: {null_before}")

# Replace whitespace-only strings with NaN
df_clean = df_clean.replace(r"^\s*$", np.nan, regex=True)

# Drop rows where the label is missing
df_clean = df_clean.dropna(subset=["Label"]).reset_index(drop=True)

# For numeric columns fill remaining NaN with column median
num_cols = df_clean.select_dtypes(include=[np.number]).columns
for col in num_cols:
    if df_clean[col].isnull().any():
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())

null_after = df_clean.isnull().sum().sum()
print(f"Total NaN after:  {null_after}")

## 3. Handle Infinite Values

In [ ]:
inf_before = df_clean.isin([np.inf, -np.inf]).sum().sum()
print(f"Infinite values before: {inf_before}")

df_clean[num_cols] = df_clean[num_cols].replace([np.inf, -np.inf], np.nan)
for col in num_cols:
    if df_clean[col].isnull().any():
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())

inf_after = df_clean.isin([np.inf, -np.inf]).sum().sum()
print(f"Infinite values after:  {inf_after}")

## 4. Standardize Labels

In [ ]:
print("Unique labels before cleanup:")
print(df_clean["Label"].unique())

df_clean["Label"] = df_clean["Label"].str.strip()

print("\nUnique labels after cleanup:")
print(df_clean["Label"].unique())

# Binary mapping: BENIGN vs ATTACK
label_map = {label: "ATTACK" if label != "BENIGN" else "BENIGN"
             for label in df_clean["Label"].unique()}
print("\nBinary label mapping:")
for k, v in sorted(label_map.items()):
    print(f"  {k} -> {v}")

## 5. Cleaned Dataset Summary

In [ ]:
print("=" * 50)
print("CLEANED DATASET SUMMARY")
print("=" * 50)
print(f"Shape:          {df_clean.shape}")
print(f"Columns:        {df_clean.shape[1]}")
print(f"Null values:    {df_clean.isnull().sum().sum()}")
print(f"Infinite values:{df_clean.isin([np.inf, -np.inf]).sum().sum()}")
print(f"Duplicate rows: {df_clean.duplicated().sum()}")
print()
print("Label distribution:")
print(df_clean["Label"].value_counts().to_string())
print("=" * 50)

df_clean.dtypes